In [ ]:
# ============================================================
# SICKLE-CELL IMAGE CLASSIFICATION
# DenseNet121 + VGG16 FEATURE-FUSION ENSEMBLE
# Includes negative + positive_unlabelled + positive_labelled
# Google Colab / TensorFlow 2.x
# ============================================================

import os
import gc
import json
import hashlib
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = False

# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

DATASET_DIR = Path(
    "/content/drive/MyDrive/Dataset_SickleCell"
)

OUTPUT_DIR = DATASET_DIR / "DenseNet121_VGG16_All_Positive_Results"
CONVERTED_DIR = OUTPUT_DIR / "converted_images"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONVERTED_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 8

HEAD_EPOCHS = 10
FINE_TUNE_EPOCHS = 15

HEAD_LEARNING_RATE = 1e-3
FINE_TUNE_LEARNING_RATE = 1e-5

TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20  # Fraction of remaining train/validation data

# Use all images in all three folders
POSITIVE_FRACTION = 1.0

CLASS_NAMES = ["negative", "positive"]
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Dataset folder:", DATASET_DIR)
print("Output folder:", OUTPUT_DIR)

# ============================================================
# 2. CHECK DATASET FOLDERS
# ============================================================

SOURCE_FOLDERS = {
    "negative": 0,
    "positive_unlabelled": 1,
    "positive_labelled": 1,
}

for folder_name in SOURCE_FOLDERS:
    folder_path = DATASET_DIR / folder_name

    if not folder_path.exists():
        raise FileNotFoundError(
            f"Required dataset folder not found: {folder_path}"
        )

print("\nDataset folders found:")
for folder_name in SOURCE_FOLDERS:
    folder_path = DATASET_DIR / folder_name
    file_count = sum(1 for p in folder_path.rglob("*") if p.is_file())
    print(f"{folder_name}: {file_count} files")

# ============================================================
# 3. CONVERT IMAGES TO VALID RGB PNG
#    - Includes all three folders
#    - Skips unreadable images
#    - Deduplicates identical decoded RGB images
#    - Detects identical pixels assigned conflicting labels
# ============================================================

records_by_hash = {}
unreadable_files = []
source_file_count = 0

SUPPORTED_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".gif", ".webp", ".tif", ".tiff"
}

print("\nReading and converting images...")

for folder_name, label in SOURCE_FOLDERS.items():

    source_dir = DATASET_DIR / folder_name
    output_class_dir = CONVERTED_DIR / folder_name
    output_class_dir.mkdir(parents=True, exist_ok=True)

    files = [
        p for p in source_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
    ]

    print(f"\nProcessing {folder_name}: {len(files)} image files")

    for idx, file_path in enumerate(files, start=1):
        source_file_count += 1

        try:
            with Image.open(file_path) as image:
                image.load()
                rgb_image = image.convert("RGB")

            width, height = rgb_image.size
            if width < 1 or height < 1:
                raise ValueError("Invalid image dimensions")

            # Hash decoded image pixels rather than file bytes
            pixel_bytes = rgb_image.tobytes()

            digest_input = (
                str(width).encode()
                + b"x"
                + str(height).encode()
                + b":"
                + pixel_bytes
            )

            image_hash = hashlib.sha256(digest_input).hexdigest()

            if image_hash in records_by_hash:
                previous = records_by_hash[image_hash]

                if previous["label"] != label:
                    raise ValueError(
                        "Identical RGB images were found in conflicting "
                        f"classes: {previous['source_folder']} and "
                        f"{folder_name}. Check these files before training."
                    )

                # Same image and same class: record duplicate provenance
                previous["source_files"].append(str(file_path))
                previous["source_folders"].add(folder_name)
                continue

            # Standardized PNG cache
            converted_path = output_class_dir / f"{image_hash}.png"
            rgb_image.save(converted_path, format="PNG")

            records_by_hash[image_hash] = {
                "image_hash": image_hash,
                "image_path": str(converted_path),
                "label": int(label),
                "class_name": CLASS_NAMES[label],
                "source_folder": folder_name,
                "source_folders": {folder_name},
                "source_files": [str(file_path)],
                "original_width": width,
                "original_height": height,
            }

        except Exception as exc:
            unreadable_files.append({
                "file": str(file_path),
                "reason": str(exc)
            })

        if idx % 100 == 0:
            print(f"  Checked {idx}/{len(files)} files")

# Build unique-image dataframe
records = []

for item in records_by_hash.values():
    row = item.copy()
    row["source_folders"] = ";".join(sorted(row["source_folders"]))
    row["source_files"] = ";".join(row["source_files"])
    records.append(row)

df = pd.DataFrame(records)

if df.empty:
    raise RuntimeError(
        "No readable images were found. Check the dataset paths and files."
    )

df.to_csv(OUTPUT_DIR / "all_unique_images_manifest.csv", index=False)

pd.DataFrame(unreadable_files).to_csv(
    OUTPUT_DIR / "unreadable_images.csv",
    index=False
)

print("\nImage conversion complete.")
print("Total source image files:", source_file_count)
print("Unique usable images:", len(df))
print("Unreadable images:", len(unreadable_files))
print("\nUnique images by class:")
print(df["class_name"].value_counts())

print("\nUnique images by source folder:")
print(
    df.groupby("source_folder")["image_hash"]
      .nunique()
      .sort_values(ascending=False)
)

# ============================================================
# 4. VERIFY BOTH CLASSES ARE PRESENT
# ============================================================

class_counts = df["label"].value_counts().to_dict()

if 0 not in class_counts or 1 not in class_counts:
    raise ValueError(
        f"Both classes are required. Current class counts: {class_counts}"
    )

if min(class_counts.values()) < 10:
    raise ValueError(
        f"Too few unique images in at least one class: {class_counts}"
    )

print("\nLabel mapping:")
print("0 = negative")
print("1 = positive (labelled + unlabelled)")

# ============================================================
# 5. STRATIFIED TRAIN / VALIDATION / TEST SPLIT
#    Exact duplicate pixel images have already been removed.
# ============================================================

train_val_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df["label"]
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=VALIDATION_SIZE,
    random_state=SEED,
    stratify=train_val_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nSplit sizes:")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

for split_name, split_df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df),
]:
    print(f"\n{split_name} class distribution:")
    print(
        split_df["label"]
        .map({0: "negative", 1: "positive"})
        .value_counts()
    )

# Save the split assignments for reproducibility
split_frames = []

for split_name, split_df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    temp = split_df.copy()
    temp["split"] = split_name
    split_frames.append(temp)

pd.concat(split_frames, ignore_index=True).to_csv(
    OUTPUT_DIR / "train_validation_test_split.csv",
    index=False
)

# ============================================================
# 6. TF.DATA INPUT PIPELINE
# ============================================================

def load_png(image_path, label):
    image_bytes = tf.io.read_file(image_path)

    # All cached images are valid RGB PNGs
    image = tf.io.decode_png(image_bytes, channels=3)

    image = tf.image.resize(
        image,
        IMAGE_SIZE,
        method="bilinear"
    )

    image = tf.cast(image, tf.float32)
    image.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 3])

    label = tf.cast(label, tf.float32)

    return image, label


def make_dataset(frame, training=False):
    paths = frame["image_path"].astype(str).to_numpy()
    labels = frame["label"].astype(np.float32).to_numpy()

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(frame),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_png,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

print("\nInput pipeline created successfully.")

# ============================================================
# 7. CLASS WEIGHTS
# ============================================================

classes = np.array([0, 1])

computed_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].to_numpy()
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, computed_weights)
}

print("Class weights:", class_weights)

# ============================================================
# 8. BUILD DENSENET121 + VGG16 FEATURE-FUSION ENSEMBLE
# ============================================================

def build_ensemble():

    inputs = keras.Input(
        shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
        name="input_image"
    )

    # Pretrained backbones
    densenet = keras.applications.DenseNet121(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
        pooling="avg"
    )

    vgg = keras.applications.VGG16(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
        pooling="avg"
    )

    densenet.trainable = False
    vgg.trainable = False

    # Each backbone receives its own expected preprocessing
    densenet_input = layers.Lambda(
        keras.applications.densenet.preprocess_input,
        name="densenet_preprocessing"
    )(inputs)

    vgg_input = layers.Lambda(
        keras.applications.vgg16.preprocess_input,
        name="vgg_preprocessing"
    )(inputs)

    dense_features = densenet(densenet_input)
    vgg_features = vgg(vgg_input)

    # Normalize each feature vector before fusion
    dense_features = layers.BatchNormalization(
        name="densenet_feature_norm"
    )(dense_features)

    vgg_features = layers.BatchNormalization(
        name="vgg_feature_norm"
    )(vgg_features)

    # Feature-level ensemble
    fused = layers.Concatenate(
        name="feature_fusion"
    )([dense_features, vgg_features])

    fused = layers.Dense(
        512,
        activation="relu",
        name="fusion_dense"
    )(fused)

    fused = layers.BatchNormalization()(fused)
    fused = layers.Dropout(0.50)(fused)

    fused = layers.Dense(
        128,
        activation="relu",
        name="classifier_dense"
    )(fused)

    fused = layers.Dropout(0.30)(fused)

    outputs = layers.Dense(
        1,
        activation="sigmoid",
        name="positive_probability"
    )(fused)

    model = keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="DenseNet121_VGG16_FeatureFusion"
    )

    return model, densenet, vgg


model, densenet_base, vgg_base = build_ensemble()

def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall")
        ]
    )


compile_model(model, HEAD_LEARNING_RATE)

model.summary()

# ============================================================
# 9. TRAINING CALLBACKS
# ============================================================

head_checkpoint = str(
    OUTPUT_DIR / "best_head_model.keras"
)

fine_checkpoint = str(
    OUTPUT_DIR / "best_finetuned_model.keras"
)


def create_callbacks(checkpoint_path, patience=4):
    return [
        keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1
        ),

        keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=patience,
            restore_best_weights=True,
            verbose=1
        ),

        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=2,
            min_lr=1e-7,
            verbose=1
        ),

        keras.callbacks.CSVLogger(
            str(OUTPUT_DIR / (
                "head_training_log.csv"
                if checkpoint_path == head_checkpoint
                else "finetune_training_log.csv"
            ))
        )
    ]


# ============================================================
# 10. PHASE 1: TRAIN THE FUSION CLASSIFIER
# ============================================================

print("\n" + "=" * 65)
print("PHASE 1: TRAINING CLASSIFICATION HEAD")
print("=" * 65)

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    class_weight=class_weights,
    callbacks=create_callbacks(head_checkpoint),
    verbose=1
)

best_head_auc = max(history_head.history.get("val_auc", [0.0]))

print("\nBest head validation AUC:", round(best_head_auc, 4))

# ============================================================
# 11. PHASE 2: FINE-TUNE THE TOP LAYERS OF BOTH BACKBONES
# ============================================================

print("\n" + "=" * 65)
print("PHASE 2: FINE-TUNING BOTH BACKBONES")
print("=" * 65)

def unfreeze_top_layers(base_model, number_of_layers):
    base_model.trainable = True

    cutoff = max(0, len(base_model.layers) - number_of_layers)

    for index, layer in enumerate(base_model.layers):
        layer.trainable = index >= cutoff

        # Keep batch-normalization layers frozen for stability
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False


# Fine-tune only the upper layers
unfreeze_top_layers(densenet_base, 30)
unfreeze_top_layers(vgg_base, 20)

# Recompile after changing trainability
compile_model(model, FINE_TUNE_LEARNING_RATE)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    class_weight=class_weights,
    callbacks=create_callbacks(fine_checkpoint),
    verbose=1
)

best_fine_auc = max(history_fine.history.get("val_auc", [0.0]))

print("\nBest fine-tuning validation AUC:", round(best_fine_auc, 4))

# ============================================================
# 12. SELECT THE MODEL USING VALIDATION AUC ONLY
#    The test set is not used to choose the model.
# ============================================================

if best_fine_auc >= best_head_auc and os.path.exists(fine_checkpoint):
    chosen_checkpoint = fine_checkpoint
    selected_phase = "fine_tuning"
else:
    chosen_checkpoint = head_checkpoint
    selected_phase = "classification_head"

if not os.path.exists(chosen_checkpoint):
    raise FileNotFoundError(
        f"Expected model checkpoint not found: {chosen_checkpoint}"
    )

model = keras.models.load_model(
    chosen_checkpoint,
    compile=False,
    safe_mode=False
)

compile_model(model, FINE_TUNE_LEARNING_RATE)

print("\nSelected model phase:", selected_phase)

# ============================================================
# 13. EVALUATE ON THE HELD-OUT TEST SET
# ============================================================

print("\n" + "=" * 65)
print("FINAL TEST-SET EVALUATION")
print("=" * 65)

test_results = model.evaluate(
    test_ds,
    return_dict=True,
    verbose=1
)

print("\nTest metrics:")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")

# Collect predictions in the same order as test_df
y_true = test_df["label"].to_numpy().astype(int)

y_probability = model.predict(
    test_ds,
    verbose=1
).ravel()

y_pred = (y_probability >= 0.5).astype(int)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

if len(np.unique(y_true)) == 2:
    auc = roc_auc_score(y_true, y_probability)
else:
    auc = float("nan")

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["negative", "positive"],
        zero_division=0
    )
)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1-score :", round(f1, 4))
print("ROC-AUC  :", round(auc, 4))

# ============================================================
# 14. SAVE METRICS AND TEST PREDICTIONS
# ============================================================

metrics = {
    "selected_phase": selected_phase,
    "best_head_validation_auc": float(best_head_auc),
    "best_finetune_validation_auc": float(best_fine_auc),
    "test_loss": float(test_results.get("loss", float("nan"))),
    "test_accuracy": float(accuracy),
    "test_precision": float(precision),
    "test_recall": float(recall),
    "test_f1_score": float(f1),
    "test_roc_auc": float(auc),
    "total_unique_images": int(len(df)),
    "training_images": int(len(train_df)),
    "validation_images": int(len(val_df)),
    "test_images": int(len(test_df)),
    "negative_images": int((df["label"] == 0).sum()),
    "positive_images": int((df["label"] == 1).sum()),
    "unreadable_source_files": int(len(unreadable_files)),
    "image_size": list(IMAGE_SIZE),
    "batch_size": int(BATCH_SIZE),
    "random_seed": int(SEED)
}

with open(OUTPUT_DIR / "test_metrics.json", "w") as file:
    json.dump(metrics, file, indent=4)

prediction_df = test_df[
    ["image_hash", "image_path", "label", "class_name", "source_folder"]
].copy()

prediction_df["true_label"] = y_true
prediction_df["predicted_label"] = y_pred
prediction_df["positive_probability"] = y_probability
prediction_df["predicted_class"] = np.where(
    y_pred == 1, "positive", "negative"
)

prediction_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

# ============================================================
# 15. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["negative", "positive"],
    yticklabels=["negative", "positive"]
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("DenseNet121 + VGG16 — Test Confusion Matrix")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# ============================================================
# 16. ROC CURVE
# ============================================================

if len(np.unique(y_true)) == 2:
    fpr, tpr, thresholds = roc_curve(y_true, y_probability)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve — Held-out Test Set")
    plt.legend(loc="lower right")
    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / "roc_curve.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

# ============================================================
# 17. TRAINING CURVES
# ============================================================

def plot_training_curves(history, prefix):
    history_data = history.history

    if "loss" in history_data:
        plt.figure(figsize=(7, 5))
        plt.plot(history_data["loss"], label="Training loss")
        if "val_loss" in history_data:
            plt.plot(history_data["val_loss"], label="Validation loss")

        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title(f"{prefix}: Loss")
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            OUTPUT_DIR / f"{prefix.lower()}_loss.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()

    if "auc" in history_data:
        plt.figure(figsize=(7, 5))
        plt.plot(history_data["auc"], label="Training AUC")
        if "val_auc" in history_data:
            plt.plot(history_data["val_auc"], label="Validation AUC")

        plt.xlabel("Epoch")
        plt.ylabel("AUC")
        plt.title(f"{prefix}: AUC")
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            OUTPUT_DIR / f"{prefix.lower()}_auc.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()


plot_training_curves(history_head, "Head_Training")
plot_training_curves(history_fine, "Fine_Tuning")

# ============================================================
# 18. SAVE FINAL MODEL
# ============================================================

FINAL_MODEL_PATH = (
    OUTPUT_DIR / "densenet121_vgg16_all_positives_ensemble.keras"
)

model.save(FINAL_MODEL_PATH)

print("\nFinal model saved to:")
print(FINAL_MODEL_PATH)

# ============================================================
# 19. SAVE A SUMMARY OF THE RUN
# ============================================================

summary_lines = [
    "DenseNet121 + VGG16 Feature-Fusion Ensemble",
    "=" * 55,
    "Classes:",
    "0 = negative",
    "1 = positive (positive_unlabelled + positive_labelled)",
    "",
    "Unique images per class:",
    str(df["class_name"].value_counts().to_dict()),
    "",
    "Split sizes:",
    f"Training: {len(train_df)}",
    f"Validation: {len(val_df)}",
    f"Testing: {len(test_df)}",
    "",
    "Selected phase:",
    selected_phase,
    "",
    "Test metrics:",
    json.dumps(metrics, indent=4),
    "",
    "Final model:",
    str(FINAL_MODEL_PATH)
]

with open(OUTPUT_DIR / "run_summary.txt", "w") as file:
    file.write("\n".join(summary_lines))

print("\n" + "=" * 65)
print("RUN COMPLETE")
print("=" * 65)
print("Results directory:", OUTPUT_DIR)
print("\nFiles created:")

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(" -", path.name)

print("\nNote: converted_images is a directory containing standardized PNGs.")
print("Original dataset folders were not modified.")

ValueError: mount failed